# Robustness Test — How Does the Model Hold Up Under Real-World Noise?
### Dataset: CICIDS-2017  ·  Runs on **Kaggle**

## The question

Our test set comes from the *same clean capture* as the training data. Real deployment is
messier: sensors drift, some features are occasionally unmeasured, traffic conditions differ
from the lab. *"How will the model perform in the real world?"* really means *"how gracefully
does it degrade when the input is no longer pristine?"*

## Why this, not hand-made synthetic flows

We deliberately do **not** invent fake attack flows. To synthesise an "attack" we would have
to assume what an attack looks like — but that is exactly what the model learned, so the test
would be circular and have no real ground truth. Instead we take **real test flows with their
real labels** and apply realistic *corruptions* of increasing severity, then watch the metric
decay. This keeps genuine ground truth while simulating the kinds of distortion a deployed
sensor introduces.

## The perturbations (each applied to the scaled test features, at rising severity)

| Perturbation | Simulates |
|---|---|
| **Gaussian noise** (σ = 0.05 … 1.0) | measurement jitter / sensor imprecision |
| **Feature scaling drift** (×0.8 … ×1.5 random per feature) | a differently-calibrated capture environment |
| **Feature dropout** (5 % … 50 % of features zeroed) | features that a real sensor failed to compute |

A model that holds up well under mild noise and degrades *gracefully* (not catastrophically)
under heavy noise is one you can trust in the field. A model that falls off a cliff at the
slightest perturbation was probably exploiting brittle, over-specific patterns.

We test the **XGBoost** model (the project's strongest) on the **multi-class** task, since the
rare classes are where brittleness would show first.

**Kaggle setup:** attach the FeatureSelection output dataset; set `IN_DIR`. GPU optional.

## 1. Imports & Config

In [ ]:
import os, json, time, warnings
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, accuracy_score
import xgboost as xgb

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.dpi']=110; plt.rcParams['savefig.bbox']='tight'

RANDOM_SEED=42; np.random.seed(RANDOM_SEED)
CLASS_NAMES=['BENIGN','Bot/Infiltration','Brute Force','DDoS','DoS','PortScan','Web Attack']
N_CLASSES=len(CLASS_NAMES)

IN_DIR='/kaggle/input/ids-selected'   # EDIT to your FeatureSelection mount path
OUT_DIR='/kaggle/working'; FIGURES_DIR=os.path.join(OUT_DIR,'figures'); os.makedirs(FIGURES_DIR,exist_ok=True)

_report=[]
def _log(t=''): _report.append(str(t)); print(t)
def _savefig(n,fig=None): p=os.path.join(FIGURES_DIR,n); (fig or plt).savefig(p,dpi=130,bbox_inches='tight'); return p
def write_report():
    p=os.path.join(OUT_DIR,'Robustness_Perturbation_Report.txt')
    open(p,'w',encoding='utf-8').write('\n'.join(_report)); print('Report ->',p)

try:
    import subprocess; subprocess.check_output(['nvidia-smi','-L']); GPU=True
except Exception: GPU=False

_log('='*70); _log('ROBUSTNESS / PERTURBATION TEST  —  CICIDS-2017')
_log(f'Generated : {datetime.now().strftime("%Y-%m-%d %H:%M")}'); _log(f'GPU : {GPU}'); _log('='*70)
print('Setup complete.')

## 2. Load Data & Train the Model (clean)
Train XGBoost on the clean training data once. Establish the clean-test baseline. All
perturbations are then applied to the *test* features only — the model never sees corrupted
data during training, exactly as in deployment.

In [ ]:
train_path=os.path.join(IN_DIR,'train_selected.parquet')
test_path =os.path.join(IN_DIR,'test_selected.parquet')
feat_path =os.path.join(IN_DIR,'selected_features.json')
for p in [train_path,test_path,feat_path]:
    if not os.path.exists(p): raise FileNotFoundError(f'{p} not found. Set IN_DIR.')
selected_features=json.load(open(feat_path))['selected_features']
train_df=pd.read_parquet(train_path); test_df=pd.read_parquet(test_path)
n_features=len(selected_features)

X_train=train_df[selected_features].values.astype(np.float32)
y_train=train_df['label_multi'].values.astype(np.int64)
X_test =test_df[selected_features].values.astype(np.float32)
y_test =test_df['label_multi'].values.astype(np.int64)

# inverse-frequency sample weights (same imbalance handling as notebook 09)
cc=np.bincount(y_train,minlength=N_CLASSES).astype(float); sw=(len(y_train)/(N_CLASSES*np.maximum(cc,1)))[y_train]

_log(''); _log('── SECTION 2 : TRAIN CLEAN MODEL ──────────────────────────')
t0=time.time()
model=xgb.XGBClassifier(objective='multi:softprob',num_class=N_CLASSES,tree_method='hist',
                        device=('cuda' if GPU else 'cpu'),n_estimators=400,max_depth=10,
                        learning_rate=0.3,random_state=RANDOM_SEED,verbosity=0)
model.fit(X_train,y_train,sample_weight=sw)
clean_pred=model.predict(X_test)
clean_f1=f1_score(y_test,clean_pred,average='macro'); clean_acc=accuracy_score(y_test,clean_pred)
_log(f'  Trained in {time.time()-t0:.1f}s')
_log(f'  CLEAN test: macro-F1={clean_f1:.4f}  acc={clean_acc:.4f}  (this is the baseline to degrade from)')

## 3. Perturbation Functions

In [ ]:
def add_noise(X, sigma, rng):
    """Gaussian noise — features are StandardScaled, so sigma is in std units."""
    return X + rng.normal(0, sigma, X.shape).astype(np.float32)

def scale_drift(X, spread, rng):
    """Multiply every feature column by a random factor in [1-spread, 1+spread]."""
    factors = (1 + rng.uniform(-spread, spread, X.shape[1])).astype(np.float32)
    return X * factors

def feature_dropout(X, frac, rng):
    """Zero out a random `frac` of features per row (simulates unmeasured features)."""
    mask = (rng.rand(*X.shape) >= frac).astype(np.float32)
    return X * mask

def evaluate(Xp):
    p = model.predict(Xp)
    return f1_score(y_test, p, average='macro'), accuracy_score(y_test, p)

print('Perturbation functions defined.')

## 4. Sweep Each Perturbation at Rising Severity

In [ ]:
rng=np.random.RandomState(RANDOM_SEED)
_log(''); _log('── SECTION 4 : PERTURBATION SWEEPS ────────────────────────')

noise_levels=[0.0,0.05,0.1,0.25,0.5,0.75,1.0]
drift_levels=[0.0,0.1,0.2,0.3,0.5]
drop_levels =[0.0,0.05,0.1,0.2,0.35,0.5]

noise_f1=[]; drift_f1=[]; drop_f1=[]
_log('  Gaussian noise (sigma in std units):')
for s in noise_levels:
    f1,_=evaluate(X_test if s==0 else add_noise(X_test,s,rng)); noise_f1.append(f1)
    _log(f'    sigma={s:<5} macro-F1={f1:.4f}')
_log('  Feature scaling drift (+/- spread):')
for s in drift_levels:
    f1,_=evaluate(X_test if s==0 else scale_drift(X_test,s,rng)); drift_f1.append(f1)
    _log(f'    spread={s:<5} macro-F1={f1:.4f}')
_log('  Feature dropout (fraction zeroed):')
for s in drop_levels:
    f1,_=evaluate(X_test if s==0 else feature_dropout(X_test,s,rng)); drop_f1.append(f1)
    _log(f'    frac={s:<5} macro-F1={f1:.4f}')

## 5. Degradation Curves & Verdict

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(21,5.5))
for ax,(x,y,lbl,xl) in zip(axes,[
    (noise_levels,noise_f1,'Gaussian noise','noise sigma (std units)'),
    (drift_levels,drift_f1,'Scaling drift','drift spread (+/-)'),
    (drop_levels, drop_f1, 'Feature dropout','fraction of features zeroed')]):
    ax.plot(x,y,'o-',lw=2,markersize=7,color='#1976D2')
    ax.axhline(clean_f1,ls='--',color='green',label=f'clean baseline {clean_f1:.3f}')
    ax.axhline(1/N_CLASSES,ls=':',color='red',label=f'chance {1/N_CLASSES:.3f}')
    ax.set_title(lbl,fontweight='bold'); ax.set_xlabel(xl); ax.set_ylabel('macro-F1')
    ax.set_ylim(0,1.02); ax.legend(fontsize=8)
    for xi,yi in zip(x,y): ax.text(xi,yi,f'{yi:.2f}',fontsize=7,va='bottom')
plt.suptitle('Real-World Robustness — graceful degradation under input corruption',fontsize=14,fontweight='bold')
plt.tight_layout(); _savefig('01_degradation_curves.png',fig); plt.show()

# verdict: how much noise before we lose half the macro-F1?
half=clean_f1/2
def first_below(levels,vals,thr):
    for l,v in zip(levels,vals):
        if v<thr: return l
    return None
_log(''); _log('='*70); _log('SECTION 5 : VERDICT'); _log('='*70)
_log(f'  Clean macro-F1 = {clean_f1:.4f}')
_log(f'  macro-F1 at noise sigma=0.25 : {noise_f1[noise_levels.index(0.25)]:.4f}'
     f'   (retains {noise_f1[noise_levels.index(0.25)]/clean_f1*100:.0f}% of clean)')
_log(f'  macro-F1 at 20% feature dropout : {drop_f1[drop_levels.index(0.2)]:.4f}'
     f'   (retains {drop_f1[drop_levels.index(0.2)]/clean_f1*100:.0f}% of clean)')
nb_=first_below(noise_levels,noise_f1,half)
_log(f'  Noise sigma needed to halve macro-F1 : {nb_ if nb_ is not None else ">1.0 (very robust)"}')
_log('')
_log('  Interpretation: a gentle initial slope = the model degrades gracefully and should')
_log('  transfer reasonably to noisier real-world capture. A cliff at small sigma would mean')
_log('  it relied on brittle, over-precise feature values.')

pd.DataFrame({'noise_sigma':noise_levels,'macro_f1':noise_f1}).to_csv(os.path.join(OUT_DIR,'robust_noise.csv'),index=False)
pd.DataFrame({'drift_spread':drift_levels,'macro_f1':drift_f1}).to_csv(os.path.join(OUT_DIR,'robust_drift.csv'),index=False)
pd.DataFrame({'dropout_frac':drop_levels,'macro_f1':drop_f1}).to_csv(os.path.join(OUT_DIR,'robust_dropout.csv'),index=False)
write_report(); print('Saved robustness CSVs + report.')